# Suite FRM — The framework

Every dialect implements the same protocols (`@mbse/expressions/Framework`): its terms are values of the framework's
`Term`, the union of the dialects' `Term`s, stored through its install, structurally traversable through forms, validatable, evaluatable and
translatable. `Framework.Terms` reads any dialect's terms by its kinds (`Terms.Kind`, `Terms.Dialect`); this suite
checks that machinery on every dialect at once, and on an extension of Patterns declared here. The dialects' own rules are
suite DIA, and translation suite TRN.

In [ ]:
import { Evaluators as V, Expressions as E, Framework } from "@mbse/expressions";
import * as Patterns from "@mbse/expressions/Dialects/Patterns";
import * as CDialect from "@mbse/expressions/Dialects/C";
import * as CppDialect from "@mbse/expressions/Dialects/Cpp";
import * as Excel from "@mbse/expressions/Dialects/Excel";
import * as Latex from "@mbse/expressions/Dialects/Latex";
import * as Matlab from "@mbse/expressions/Dialects/Matlab";
import * as Python3 from "@mbse/expressions/Dialects/Python3";
import * as SystemVerilog from "@mbse/expressions/Dialects/SystemVerilog";
import { Domains as D, Errors as FE, Evaluators as FV, Partials as FP, Symbolics as SY, Terms as F, Translators as FT } from "@mbse/expressions/Framework";
import * as Translators from "@mbse/expressions/Translators";
import { Builders, Errors, JSON, Repr, Validators, YAML } from "@mbse/schemas/Framework";
import { assert, conformance_problems, equal, form_equal, map, raises, same } from "./support.js";

const BD = Patterns.Domains;
const [P, M, X, L, C, CPP, SV] = [Python3.Expressions, Matlab.Expressions, Excel.Expressions, Latex.Expressions,
  CDialect.Expressions, CppDialect.Expressions, SystemVerilog.Expressions];
const { KeyError, NotImplementedError, ValueError } = Errors;

const store: any = new Builders.Store();
for (const install of [Framework.install, Patterns.install, Python3.install, Matlab.install, Excel.install, Latex.install,
  CDialect.install, CppDialect.install, SystemVerilog.install]) {
  store.transaction(install);
}
const Term: any = store.registered("Expressions.Framework.Term");

/** The term value `spec` writes, built in a transaction. */
function term(spec: unknown, dialect: F.Dialect = E.DIALECT): any {
  return store.transaction(() => store.Expressions.Framework.Term(dialect.spec(spec)));
}

const person = E.variable("person");
const basic = E.let_("a", person.age, E.variable("a").ge(18n).and_(person.has("email")));
const samples: Record<string, any> = { // one term per dialect, with every kind
  Patterns: term(E.operation("and", basic, E.list_(1n, E.mapping(["k", person.items]).item("k")).all( // a quantifier too
    "x", E.variable("x").gt(0n)))),
  Python3: term(P.import_("numpy", P.importfrom("math", "floor", P.let_("a", P.name("person"), P.ifexp( // every kind
    P.boolop("or", P.boolop("and", P.unaryop("not", P.compare("==", P.attribute(P.name("a"), "z"), 1n, ["<", 2n])),
      P.unaryop("-", P.attribute(P.name("a"), "w"))), P.binop("*", 2n, 3n), P.namedexpr("q", 0n)),
    P.call("np.add", P.attribute(P.name("a"), "x"), P.subscript(P.name("a"), P.tuple_(P.slice_(1n, null, -1n), P.constant(P.ELLIPSIS))),
      P.starred(P.list_(1n)), P.keyword("k", P.dict_(["k", 2n], [null, P.name("a")]))),
    P.call("f", P.generatorexp(P.subscript(P.name("p"), 0n), P.comprehension("p", P.attribute(P.name("a"), "z"), [P.name("p")])),
      P.listcomp(P.name("p"), P.comprehension(P.tuple_(P.name("p"), P.starred(P.name("r"))), P.name("a"), [], true)),
      P.setcomp(P.name("p"), P.comprehension("p", P.name("a"))), P.dictcomp(P.name("p"), null, P.comprehension("p", P.name("a"))),
      P.set_(P.constant(null), P.constant(new P.Complex(0, 1))), P.lambda_(P.name("b"), [P.arg("b", 1n)]), P.yield_(),
      P.yieldfrom(P.name("a")), P.await_(P.name("a")), P.concatenatedstring("x", P.joinedstr(["y", P.formattedvalue(P.name("a"), "r")])),
      P.templatestr([P.interpolation(P.name("a"))]), P.parenthesized(P.name("a"))))),
  "f"), "np"), P.DIALECT),
  Matlab: term(M.import_("pkg.*", M.binary("&&", M.unary("~", M.field(M.identifier("person"), "age")),
    M.call("isfield", "x", M.index(M.array(M.arrayfun("p", M.identifier("person"), M.identifier("p")), 2n), 1n)))), M.DIALECT),
  Excel: term(X.let_("a", X.field(X.name("person"), "age"), X.function("AND", X.infix(">=", X.name("a"), X.cell("A1")),
    X.prefix("-", X.function("SUM", X.map_("p", X.name("a"), X.infix("*", X.name("p"), 2n)))))), X.DIALECT),
  Latex: term(L.where("a", L.member(L.symbol("person"), "age"), L.binary("\\land", L.binary("\\geq", L.symbol("a"), 18n), L.binary(
    "\\lor", L.unary("\\lnot", L.function("has", L.symbol("person"), "email")), L.binary(">", L.frac(L.symbol("a"), 2n), 1.5)))), L.DIALECT),
  C: term(C.conditional(C.binary("&&", C.unary("!", C.member(C.identifier("person"), "a")), C.binary( // every kind
    ">", C.subscript(C.member(C.identifier("person"), "b", "->"), 0n), C.constant(1.5, "float"))),
    C.cast("uint8_t", C.call("f", 1n)), C.constant(5n, "unsigned int")), C.DIALECT),
  Cpp: term(CPP.conditional(CPP.binary("&&", CPP.unary("!", CPP.member(CPP.identifier("person"), "a")), CPP.binary( // C's kinds
    ">", CPP.subscript(CPP.member(CPP.identifier("person"), "b", "->"), 0n), CPP.constant(1.5, "float"))),
    CPP.cast("uint8_t", CPP.call("f", 1n)), CPP.constant(5n, "unsigned int")), CPP.DIALECT),
  SystemVerilog: term(SV.binary("->", SV.inside(SV.member(SV.identifier("person"), "a"), 1n, SV.span(2n, 5n)), SV.conditional( // every kind
    SV.unary("!", SV.select(SV.identifier("person"), 0n)), SV.concatenation(SV.replication(2n, SV.vector("10")), SV.range_(SV.identifier("person"), 3n, 0n)),
    SV.cast("byte", SV.call("$clog2", SV.iterate(SV.method(SV.identifier("person"), "unique"), "sum", "p", SV.identifier("p")))))), SV.DIALECT),
};
const DIALECTS = [E.DIALECT, P.DIALECT, M.DIALECT, X.DIALECT, L.DIALECT, C.DIALECT, CPP.DIALECT, SV.DIALECT];
const EVALUATORS = new Map<F.Dialect, unknown>([[E.DIALECT, V.OfAny], [P.DIALECT, Python3.Evaluators.OfAny], [M.DIALECT, Matlab.Evaluators.OfAny],
  [X.DIALECT, Excel.Evaluators.OfAny], [C.DIALECT, CDialect.Evaluators.OfAny], [CPP.DIALECT, CppDialect.Evaluators.OfAny], [SV.DIALECT, SystemVerilog.Evaluators.OfAny]]); // Latex has none
const INVALID = new Set(["error", "missing"]); // the framework's kinds of invalid syntax, which every dialect's `Term` holds

<a id="FRM-01"></a>
## FRM-01 · Every dialect, expression, evaluator and translator implements its protocol

In [ ]:
assert([...EVALUATORS.values()].every((evaluate) => typeof evaluate === "function"));
for (const dialect of DIALECTS) {
  assert(dialect instanceof F.Dialect && [...dialect.kinds().values()].every((kind) => kind instanceof F.Kind));
  for (const node of F.walk(samples[dialect.name()])) { // a term of its own union, nested in its kinds or in shared ones
    assert(F.isTerm(node) && F.dialectOf(node) === dialect.union && dialect.kinds().has(F.kindOf(node)[0]));
  }
}
assert(same(DIALECTS.map((d) => d.name()), ["Patterns", "Python3", "Matlab", "Excel", "Latex", "C", "Cpp", "SystemVerilog"]));
assert(Repr.repr(E.DIALECT) === "<dialect Patterns>" && same(DIALECTS.map((d) => d.tag), Term.branches.map((b: any) => b.name)));
assert(same(DIALECTS.map((d) => d.union), Term.branches.map((b: any) => b.schema.name))); // the union of the dialects' `Term`s
assert(same([...X.DIALECT.kinds().keys()], ["literal", "variable", "cell", "let", "call", "infix", "prefix", "field", "map", "error",
  "missing"]));
for (const dialect of DIALECTS) { // each union's branches are its dialect's kinds, named plainly; shared kinds, the framework's
  const union = store.registered(dialect.union);
  assert(equal(union.branches.map((b: any) => b.name).sort(), [...dialect.kinds().keys()].sort()), dialect.name());
  const shared = union.branches.filter((b: any) => b.schema.name.startsWith("Expressions.Framework.")).map((b: any) => b.name);
  assert(shared.every((tag: string) => ["literal", "call", "variable", "let", "list", "mapping", "error", "missing"].includes(tag))
    && [...INVALID].every((tag) => shared.includes(tag)));
}
const [C_UNION, CPP_UNION] = [store.registered(C.DIALECT.union), store.registered(CPP.DIALECT.union)];
assert(same(C_UNION.branches.map((b: any) => b.name), CPP_UNION.branches.map((b: any) => b.name))); // the same kinds, each its own
const ownKinds = (union: any) => union.branches.map((b: any) => b.schema.name).filter((name: string) => !name.startsWith("Expressions.Framework."));
assert(same(ownKinds(C_UNION), ["Expressions.C.Constant", "Expressions.C.Unary", "Expressions.C.Binary", "Expressions.C.Conditional",
  "Expressions.C.Cast", "Expressions.C.Member", "Expressions.C.Subscript"]));
assert(same(ownKinds(CPP_UNION), ["Expressions.Cpp.Constant", "Expressions.Cpp.Unary", "Expressions.Cpp.Binary",
  "Expressions.Cpp.Conditional", "Expressions.Cpp.Cast", "Expressions.Cpp.Member", "Expressions.Cpp.Subscript"]));
assert(same([...E.DIALECT.kinds().keys()], ["literal", "call", "variable", "let", "list", "mapping", "quantifier", "error", "missing"]));
assert(same([...E.DIALECT.kinds().values()].map((kind) => kind.role), [
  F.LITERAL, F.APPLICATION, F.REFERENCE, F.BINDING, F.APPLICATION, F.APPLICATION, F.QUANTIFIER, F.INVALID, F.INVALID]));
for (const domain of [D.Anything, new D.OfTypes("int", BigInt), new D.OfValues("odd", (v) => Number(v) % 2 === 1), new D.OfUnion(D.Anything)]) {
  assert(conformance_problems(domain.constructor as any, "Domain").length === 0);
}
for (const signature of [new D.Function([], D.Anything), new D.Same(1, [D.Anything]), new D.Overloaded(new D.Function([], D.Anything))]) {
  assert(conformance_problems(signature.constructor as any, "Signature").length === 0);
}
for (const translator of Translators.TRANSLATORS) {
  assert(translator instanceof FT.Pairwise && translator.inverse() instanceof FT.Pairwise
    && conformance_problems(FT.Pairwise, "Translator").length === 0);
}

<a id="FRM-02"></a>
## FRM-02 · Forms and `make` are inverses, for every kind of every dialect

In [ ]:
for (const dialect of DIALECTS) {
  const kinds = new Set<string>();
  for (const node of F.walk(samples[dialect.name()])) {
    const form = F.form(node);
    kinds.add(form.kind);
    const made = term(dialect.make(form));
    assert(F.same(made, node) && F.form(made).kind === form.kind && made !== node);
  }
  assert(equal([...kinds].sort(), [...dialect.kinds().keys()].filter((tag) => !INVALID.has(tag)).sort()), dialect.name()); // every valid kind
}
assert(form_equal(F.form(term(3n)), new F.Form("literal", { value: 3n })));
assert(form_equal(F.form(term(E.let_("x", null, null))), new F.Form("let", { name: "x" }, [null, null]))); // empty slots are null
assert(form_equal(F.form(term((u: any) => u.call((c: any) => c))), new F.Form("call")));
raises(ValueError, () => E.DIALECT.make(new F.Form("constant")), "Patterns has no kind 'constant'");
raises(ValueError, () => term(E.DIALECT.make(new F.Form("literal", { name: "x" }))), "a literal has no attribute 'name'");
raises(ValueError, () => term(E.DIALECT.make(new F.Form("let", { name: "x" }, [basic]))), "a let takes 2 arguments, got 1");
raises(ValueError, () => term(E.DIALECT.make(new F.Form("quantifier", { name: "x", quantifier: "all" }, [1n, 2n, 3n]))),
  "a quantifier takes 2 arguments, got 3");
const spelled = term((u: any) => u.literal((r: any) => r.text("08").problems(["leading zeros"]))); // a container of scalars too
assert(F.same(term(E.DIALECT.make(F.form(spelled))), spelled));
let made = term(E.DIALECT.make(new F.Form("call", { callable: "f" }, [E.variable("g"), 1n, 2n]))); // specs: writers, natives
assert(same(F.form(made).arguments.map((a) => F.kindOf(a)[0]), ["variable", "literal", "literal"]));
raises(ValueError, () => F.kindOf(term((u: any) => u)), "a term holds no dialect"); // nothing written: no branch of Term is selected
const alone = store.transaction(() => store.Expressions.Patterns.Term((u: any) => u)); // a value of a dialect's `Term`, alone
assert(F.isTerm(alone) && F.dialectOf(alone) === E.DIALECT.union);
raises(ValueError, () => F.kindOf(alone), "a term holds no kind");
raises(ValueError, () => term(P.DIALECT.make(new F.Form(P.CALL)), P.DIALECT), "a call takes at least 1 arguments, got 0"); // its function, then its arguments
made = term(P.DIALECT.make(new F.Form(P.CALL, {}, [P.name("f"), P.constant(1n), 2n])), P.DIALECT); // a native: Python's constant
assert(F.nameOf(F.form(made).arguments[0]) === "f" && F.form(made).arguments.slice(1).every((a, i) =>
  form_equal(F.form(a), new F.Form(P.LITERAL, { value: BigInt(i + 1) }))));
assert(F.kindOf(term(1n, M.DIALECT))[0] === "literal" && F.kindOf(term(true, C.DIALECT))[0] === "constant"); // its literal kind
raises(TypeError, () => E.DIALECT.spec([1n]), "expected a term, a writer or a native value, got [1]"); // no literal kind holds these
raises(TypeError, () => X.DIALECT.spec(new Uint8Array([120])), "an Excel literal cannot hold a bytes");

<a id="FRM-03"></a>
## FRM-03 · `walk`, `fold` and `same` traverse through forms alone

In [ ]:
const shared = E.variable("x");
const pair = term(E.operation("f", shared, shared));
assert(same([...F.walk(pair)].map((e) => F.form(e).kind), ["call", "variable", "variable"])); // a nested value: each visited
assert(equal([...F.walk(term(basic))].map((e) => F.form(e).attributes).slice(0, 3), map([{ name: "a" }, { callable: "get" }, { name: "person" }])));
assert(F.fold(pair, (node, results: number[]) => 1 + results.reduce((a, b) => a + b, 0)) === 3);
const calls: unknown[] = [];
F.fold(pair, (node) => calls.push(node));
assert(calls.length === 3); // once per term
assert(equal(F.fold(term(E.let_("x", null, 1n)), (node, results) => results), [null, []]));
for (const [a, b, expected] of [[1n, 1n, true], [1n, 1.0, false], [NaN, NaN, true], [-0, 0, false],
  [0, 0, true], ["x", "x", true], [new Uint8Array([120]), new Uint8Array([120]), true], [new Uint8Array([120]), new Uint8Array([121]), false]] as const) {
  assert(F.same(term(a), term(b)) === expected, `${String(a)} ${String(b)}`);
}
assert(F.same(term(basic), term(E.DIALECT.make(F.form(term(basic))))) && F.same(term(E.let_("x", null, null)), term(E.let_("x", null, null))));
assert(!F.same(term(E.let_("x", null, null)), term(E.let_("x", 1n, 2n))) && !F.same(term(1n), term(E.variable("x"))));
assert(!F.same(term(E.operation("f", 1n)), term(E.operation("f", 1n, 2n))));
assert(!F.same(term(E.operation("f")), term((u: any) => u.call((c: any) => c)))); // attributes differ

<a id="FRM-04"></a>
## FRM-04 · Every dialect's install adds its union of its kinds to `Term`, and its terms round-trip

In [ ]:
for (const dialect of DIALECTS) {
  assert(Term.branches.some((b: any) => b.name === dialect.tag));
  const sample = samples[dialect.name()];
  for (const [encode, decode] of [[JSON.ToJSON(store), JSON.FromJSON(store)], [YAML.ToYAML(store), YAML.FromYAML(store)]] as [any, any][]) {
    const back = decode(Term, encode(Term, sample));
    assert(F.same(back, sample) && dialect.validate(back, ["person"]).length === 0);
  }
}
// The framework reads a kind's slots, variadic arguments and attributes from its schema: a slot holds Term, or a
// dialect's union.
const of = new Map<string, Map<string, any>>(DIALECTS.map((d) => [d.name(),
  new Map<string, any>(store.registered(d.union).branches.map((b: any) => [b.name, b.schema]))]));
const kindSchemas = of.get("Patterns") as Map<string, any>;
assert(equal(F._parts(kindSchemas.get("let")), [["value", "body"], null, ["name"]]));
assert(equal(F._parts(kindSchemas.get("call")), [[], "arguments", ["callable"]]));
assert(equal(F._parts(kindSchemas.get("quantifier")), [["collection", "body"], null, ["name", "quantifier"]]));
assert(equal(F._parts(kindSchemas.get("literal")), [[], null, ["value", "text", "problems"]]));
assert(equal(F._parts(kindSchemas.get("list")), [[], "elements", []]) && equal(F._parts(kindSchemas.get("mapping")), [[], "entries", []]));
assert(equal(F._parts(kindSchemas.get("error")), [[], "terms", ["text"]]) && equal(F._parts(kindSchemas.get("missing")), [[], null, ["kind"]]));
assert(equal(F._members(kindSchemas.get("mapping")), ["key", "value"]) && F._members(kindSchemas.get("list")).length === 0
  && F._members(kindSchemas.get("let")).length === 0);
for (const dialect of DIALECTS.slice(1)) {
  for (const tag of dialect.kinds().keys()) { // a dialect's own kinds are its schemas, named by it
    const name: string = of.get(dialect.name())?.get(tag).name;
    assert(name.startsWith(`Expressions.${dialect.name()}.`) || name.startsWith("Expressions.Framework."), tag);
  }
}
const python3 = of.get("Python3") as Map<string, any>;
assert(same([...P.DIALECT.kinds().keys()].map((tag) => python3.get(tag).name).slice(-10), [
  "Expressions.Python3.Compare", "Expressions.Python3.BoolOp", "Expressions.Python3.NamedExpr", "Expressions.Python3.IfExp",
  "Expressions.Python3.Lambda", "Expressions.Framework.Let", "Expressions.Python3.Import", "Expressions.Python3.ImportFrom",
  "Expressions.Framework.Error", "Expressions.Framework.Missing"]));
assert(python3.get(P.LITERAL).refines === store.registered("Expressions.Framework.Literal")); // the framework's, refined
assert(equal(F._parts(python3.get(P.CALL)), [["func"], "args", ["keywords"]])); // a slot, variadic arguments, then helpers
assert(equal(F._helpers(python3.get(P.CALL)), ["keywords"]) && equal(F._helpers(python3.get(P.BINOP)), []));
assert(equal(F._parts(python3.get(P.GENERATOREXP)), [["elt"], null, ["generators"]]));
assert(equal(F._parts(python3.get(P.DICT)), [[], "items", []]) && equal(F._members(python3.get(P.DICT)), ["key", "value"]));
assert(equal(F._parts(of.get("Matlab")?.get("binary")), [["left", "right"], null, ["operator"]]));
assert(equal(F._parts(of.get("Excel")?.get("cell")), [[], null, ["address", "sheet", "book"]]));
assert(python3.get(P.CALL).properties.get("func").schema === store.registered(P.DIALECT.union)); // its own union
assert(of.get("C")?.get("binary").properties.get("left").schema === store.registered(C.DIALECT.union)); // C's kinds hold C's terms
assert(of.get("Cpp")?.get("binary").properties.get("left").schema === store.registered(CPP.DIALECT.union)); // and C++'s, C++'s
// Each dialect's holder, the root of an expression tree: a reference object refining the framework's Expression, its
// root narrowed to the dialect's Term. It validates, holds a term of its dialect, round-trips, prints (FRM-11), and HasRoot,
// which links the framework's Expression, links it.
const FrameworkExpression: any = store.registered("Expressions.Framework.Expression");
const holders = new Map<F.Dialect, any>();
for (const dialect of DIALECTS) {
  const Holder: any = store.registered(`Expressions.${dialect.name()}.Expression`);
  assert(Holder.ref && Holder.refines === FrameworkExpression && Holder.validate().length === 0 && F.isExpression(Holder));
  assert(Holder.properties.get("root").schema === store.registered(dialect.union)); // narrowed from Expression's Term
  const sample = samples[dialect.name()];
  const [held, linked] = store.transaction(() => {
    const held = store.Expressions[dialect.name()].Expression((e: any) => e.root(dialect.spec(sample)));
    const linked = store.Schemas.OfExpression((s: any) => s.name(`frm.${dialect.name()}`).expression(
      (a: any) => a.insert((e: any) => e.expression(held))));
    return [held, linked];
  });
  holders.set(dialect, held);
  assert(F.same(F.term(held), sample) && F.dialectOf(F.term(held)) === dialect.union);
  assert(Validators.Validate(store)(Holder, held).length === 0 && dialect.validate(held, ["person"]).length === 0);
  assert(F.same((JSON.FromJSON(store)(Holder, JSON.ToJSON(store)(Holder, held)) as any).root, held.root));
  assert(same([...(new Builders.OfExpression.Builder(linked, false, store) as any).expression].map((entry: any) => F.same(entry.expression, held)), [true]));
}
raises(Error, () => store.transaction(() => store.Expressions.Python3.Expression((e: any) => e.root(E.list_(1n).all("p", true)))),
  "'quantifier' is not a member of a union value"); // its root holds its dialect's terms alone

<a id="FRM-05"></a>
## FRM-05 · Validation follows each kind's role, in every dialect

In [ ]:
assert(DIALECTS.every((d) => d.validate(samples[d.name()], ["person"], true).length === 0));
assert(same(E.DIALECT.validate(term(E.operation("add", 1n))), ["add takes 2 arguments, got 1"]));
assert(same(E.DIALECT.validate(term(E.operation("not", E.variable("x")))), ["argument 0: variable 'x' is not bound"]));
assert(same(E.DIALECT.validate(term(E.mapping([E.variable("k"), 1n], ["b", E.list_(E.variable("x"))]))), [
  "key 0: variable 'k' is not bound", "value 1: argument 0: variable 'x' is not bound"])); // a mapping's terms, by entry
assert(same(E.DIALECT.validate(term((u: any) => u.mapping((m: any) => m.entries([(a: any) => a.as_object((o: any) => o.key(E.literal(1n)))])))), [
  "a mapping needs a value 0"]));
assert(same(E.DIALECT.validate(term(E.let_("a", E.variable("a"), E.variable("a")))), ["value: variable 'a' is not bound"]));
assert(same(E.DIALECT.validate(term((u: any) => u.let((r: any) => r.name("x").value(E.literal(1n))))), ["a let needs a body"]));
for (const [spec, bound, problems] of [
  [P.subscript(P.name("t"), "x"), [], ["value: variable 't' is not bound"]],
  [P.call("f", 1n), [], ["func: variable 'f' is not bound"]], // a slot, then variadic arguments
  [P.call(P.name("f"), P.name("x")), ["f"], ["argument 0: variable 'x' is not bound"]],
  [P.call(P.name("f"), P.keyword("k", P.name("x"))), ["f"], ["keywords 0 value: variable 'x' is not bound"]], // a helper's
  [(u: any) => u[P.CALL]((r: any) => r.func(P.name("f")).keywords([(a: any) => a.as_object((o: any) => o.arg("k"))])),
    ["f"], ["a call needs a keywords 0 value"]],
  [P.listcomp(P.name("y"), P.comprehension("y", P.name("y"))), [], ["generators 0 iter: variable 'y' is not bound"]],
  [P.lambda_(P.name("a"), ["a", P.arg("b", P.name("a"))]), [], ["args args 1 default_value: variable 'a' is not bound"]],
  [P.subscript(P.name("t"), P.slice_()), ["t"], []], // a kind's optional slots may be empty
  [P.import_("numpy", P.name("np"), "np"), [], []], // an import binds a name; its alias is optional
  [P.import_("numpy", P.name("np")), [], ["body: variable 'np' is not bound"]],
  [(u: any) => u[P.IMPORT]((r: any) => r.module("numpy")), [], ["an import needs a body"]],
  [P.name("not a name"), [], ["a name must be an identifier, got 'not a name'"]], // a kind's own check
] as [unknown, string[], string[]][]) {
  assert(same(P.DIALECT.validate(term(spec, P.DIALECT), bound), problems));
}
assert(X.DIALECT.validate(term(X.cell("A1"), X.DIALECT)).length === 0
  && X.DIALECT.validate(term(X.cell("B2", "Sheet1"), X.DIALECT), [], true).length === 0); // cells need no binding
assert(same(M.DIALECT.validate(term((u: any) => u.field((r: any) => r), M.DIALECT)), ["a field needs a name", "a field needs a value"]));
assert(same(M.DIALECT.validate(term((u: any) => u.binary((r: any) => r.operator("&&").left(M.constant(1n))), M.DIALECT)), ["a binary needs a right"]));
assert(M.DIALECT.validate(term(M.binary("**", 1n, 2n), M.DIALECT)).length === 0
  && same(M.DIALECT.validate(term(M.binary("**", 1n, 2n), M.DIALECT), [], true), ["'**' is not a core operation"]));
assert(M.DIALECT.validate(term(M.field(M.identifier("s"), "any name"), M.DIALECT), ["s"], true).length === 0); // a field's vocabulary is open
assert(same(X.DIALECT.validate(term(X.function("NOT", true, false), X.DIALECT)), ["NOT takes 1 arguments, got 2"]));
assert(X.DIALECT.validate(term(X.function("AND", true), X.DIALECT)).length === 0); // AND: any number
assert(same(X.DIALECT.validate(term(X.let_("a", X.name("a"), X.name("a")), X.DIALECT)), ["value: variable 'a' is not bound"]));
assert(same(X.DIALECT.validate(term((u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(new Uint8Array([120])))), X.DIALECT)),
  ["an Excel literal cannot hold a bytes"])); // the dialect's own check of the framework's literal
assert(same(M.DIALECT.validate(term((u: any) => u.call((r: any) => r.callable(7n)), M.DIALECT)), ["a call's callable must be a str, got int"]));

<a id="FRM-06"></a>
## FRM-06 · Domains and signatures

In [ ]:
{
  const [Int, Float, Str] = [new D.OfTypes("int", BigInt), new D.OfTypes("float", Number), new D.OfTypes("str", String)];
  const Odd = new D.OfValues("odd", (v) => Number(v) % 2 === 1);
  const Number_ = new D.OfUnion(Int, Float);
  assert(Int.contains(1n) && !Int.contains(true) && !Int.contains(1.0) && String(Int) === "int");
  assert(new D.OfTypes("bool", Boolean).contains(true) && !new D.OfTypes("bool", Boolean).contains(1n));
  assert(Str.contains("a") && !Str.contains(new Uint8Array([97])));
  assert(Number_.name() === "int | float" && String(Number_) === "int | float" && Number_.contains(1.5) && !Number_.contains("1"));
  assert(Number_.includes(Int) && Number_.includes(new D.OfUnion(Float, Int)) && !Int.includes(Number_));
  assert(new D.OfTypes("num", BigInt, Number).includes(Number_) && !Number_.includes(Str) && !Number_.includes(D.Anything));
  assert(Odd.contains(3n) && Odd.includes(Odd) && !Odd.includes(Int) && Odd.includes(new D.OfUnion(Odd)) && String(Odd) === "odd");
  assert(D.Anything.includes(Number_) && D.Anything.contains(null) && String(D.Anything) === "any" && D.Anything.name() === "any");
  assert(D.overlaps(Int, D.Anything) && D.overlaps(Number_, Int) && !D.overlaps(Int, Str));
  assert(new D.OfTypes("list", Array).contains([1n]) && !new D.OfTypes("list", Array).contains("x")); // a class, by instance
  const f = new D.Function([Int, Str], Float);
  assert(f.arity() === 2 && f.result([Int, Str]) === Float && f.result([D.Anything, Str]) === Float);
  assert(f.result([Str, Str]) === null && f.result([Int]) === null && f.describe() === "(int, str) -> float");
  assert(f.exact([Int, Str]) && !f.exact([D.Anything, Str]));
  const sameOf = new D.Same(2, [Int, Float]);
  assert(sameOf.result([Int, Int]) === Int && sameOf.result([Int, Float]) === null && sameOf.result([Int]) === null);
  assert(sameOf.result([D.Anything, D.Anything])?.name() === "int | float" && sameOf.result([D.Anything, Float]) === Float);
  assert(sameOf.describe() === "(T, T) -> T for T in int | float" && new D.Same(1, [Int], Str).describe() === "(T) -> str for T in int");
  assert(new D.Same(2, [Int, Float], Str).result([D.Anything, D.Anything]) === Str); // one result, however many match
  assert(new D.Opaque().arity() === -1 && new D.Opaque(Int).result([Str, Str, Str]) === Int && new D.Opaque().describe() === "(...) -> any");
  const either = new D.Either(2);
  assert(either.arity() === 2 && either.result([Int, Int]) === Int && either.result([Int, Str])?.name() === "int | str");
  assert(either.result([Int]) === null && either.describe() === "(T, T) -> T");
  const anyNumber = new D.Either(-1); // Python's `and` of any number of values
  assert(anyNumber.result([Int, Str, Int])?.name() === "int | str" && anyNumber.result([]) === null && anyNumber.describe() === "(T, ...) -> T");
  assert(sameOf.exact([Int, Int]) && !sameOf.exact([Int, Float]) && !sameOf.exact([Int]));
  const over = new D.Overloaded(new D.Same(2, [Int]), new D.Function([Number_, Number_], Float));
  assert(over.arity() === 2 && over.result([Int, Int]) === Int && over.result([Int, Float]) === Float);
  assert(over.result([D.Anything, Int])?.name() === "int | float" && over.result([Str, Str]) === null);
  assert(over.describe() === "(T, T) -> T for T in int | (int | float, int | float) -> float");
}

<a id="FRM-07"></a>
## FRM-07 · `infer` finds an expression's domain from its vocabulary's signatures

In [ ]:
function infer(spec: unknown, environment: Record<string, any> = {}) {
  return E.DIALECT.infer(term(spec), environment);
}

assert(infer(basic, { person: BD.Object }) === BD.Bool);
assert(infer(E.literal(1n).add(2n)) === BD.Int && infer(E.literal(1.5).neg()) === BD.Float);
assert(infer(person.age, { person: BD.Object }) === D.Anything);
assert(infer(person.age.add(1n), { person: BD.Object }) === BD.Int); // only int + int applies
assert(M.DIALECT.infer(term(M.call("dist", 1n, 2n), M.DIALECT)) === D.Anything); // an extension: nothing is known of it
raises(ValueError, () => infer(E.operation("contains", 1n, 2n)),
  "cannot infer the domain of an invalid expression: 'contains' is not an operation Patterns defines"); // Patterns' operations are the ones it defines
assert(infer(E.list_(1n, 2n)).name() === "list(int)" && infer(E.list_(1n, "a")).name() === "list(int | str)");
assert(infer(E.list_()).name() === "list(any)" && infer(E.list_(1n, 2n).count()) === BD.Int);
assert(infer(E.mapping(["a", 1.5])).name() === "keyed(str, float)" && infer(E.mapping()).name() === "keyed(any, any)");
assert(infer(E.list_(1n, 2n).item(0n)) === BD.Int && infer(E.list_(1n, 2n).all("x", E.variable("x").gt(0n))) === BD.Bool);
assert(BD.LIST.describe() === "(T, ...) -> list(T)" && BD.MAPPING.describe() === "(K, T, ...) -> keyed(K, T)" && BD.LIST.arity() === -1);
assert(infer(E.let_("a", 1n, E.let_("b", "s", E.variable("a")))) === BD.Int);
raises(TypeError, () => infer(E.literal(1n).add(1.5)), "add cannot take (int, float); it takes (T, T) -> T for T in int | float");
raises(TypeError, () => infer(E.literal(1n).has("x")), "has cannot take (int, str); it takes (object | record, str) -> bool");
raises(ValueError, () => infer(person), "cannot infer the domain of an invalid expression: variable 'person' is not bound");
assert(BD.of(term(1n)) === BD.Object && BD.of(new Uint8Array()) === BD.Bytes); // a term value is an object
raises(TypeError, () => BD.of([]), "a list is not a value of the Patterns dialect");
assert(M.DIALECT.infer(samples.Matlab, { person: D.Anything }).name() === "logical");
assert(X.DIALECT.infer(samples.Excel, { person: D.Anything }).name() === "logical");
assert(L.DIALECT.infer(samples.Latex, { person: D.Anything }).name() === "truth");
assert(P.DIALECT.infer(samples.Python3, { person: D.Anything }) === D.Anything); // a call gives what is not known
assert(P.DIALECT.infer(term(P.boolop("and", 1n, "a"), P.DIALECT)).name() === "int | str"); // Python's and gives an operand
assert(P.DIALECT.infer(term(P.import_("math", P.binop("+", 1n, true)), P.DIALECT)).name() === "int");

<a id="FRM-08"></a>
## FRM-08 · The interpreter evaluates by role and raises what validation reports

In [ ]:
let interpreter = new FV.Interpreter(E.DIALECT, new Map<string, any>([["call", new Map()], ["quantifier", new Map()]]));
raises(TypeError, () => interpreter.run(3n), "not a term: 3");
raises(ValueError, () => interpreter.run(term(E.literal(null))), "a literal needs a value");
raises(KeyError, () => interpreter.run(term(E.variable("x"))), "variable 'x' is not bound");
raises(NotImplementedError, () => interpreter.run(term(E.literal(true).and_(false))), "'and' is not an operation Patterns defines");
raises(NotImplementedError, () => new FV.Interpreter(C.DIALECT, new Map()).run(term(C.constant(5n, "unsigned int"), C.DIALECT)),
  "this evaluator has no values of unsigned int"); // a kind's own domain: C's type
raises(TypeError, () => new FV.Interpreter(E.DIALECT, new Map([["call", new Map([["not", null as any]])]])).run(term(E.operation("not", 1n, 2n))),
  "not takes 1 arguments, got 2");
raises(ValueError, () => V.OfAny(term(E.let_("a", 1n, null))), "a let needs a body");
raises(ValueError, () => V.OfAny(term(E.quantifier("all", "x", null, true))), "a quantifier needs a collection");
assert(new FV.Interpreter(E.DIALECT, new Map([["call", new Map()]]), { extension: (operator) => operator })
  .run(term(E.operation("dist", 1n, 2n))) === "dist"); // operators outside the vocabulary go to the extension

const scope = new SY.Variables({ x: 1n });
let x: any = term(E.variable("x"));
assert(conformance_problems(SY.Variables, "Scope").length === 0 && scope.lookup(x) === 1n);
const inner = scope.bind("x", 2n);
assert(inner.lookup(x) === 2n && scope.lookup(x) === 1n); // bind makes a new scope
raises(KeyError, () => scope.lookup(term(E.variable("y"))), "variable 'y' is not bound");
assert(V.OfAny(x, inner) === 2n); // an evaluator takes a scope as well as an object of variables

<a id="FRM-09"></a>
## FRM-09 · Symbolics: the names an expression needs, and the imports it declares

In [ ]:
const sorted = (names: Set<string>) => [...names].sort();
assert(same(sorted(SY.free(term(basic), E.DIALECT)), ["person"]) && SY.free(term(1n), E.DIALECT).size === 0);
assert(same(sorted(SY.free(term(E.let_("a", E.variable("a"), E.variable("a"))), E.DIALECT)), ["a"])); // the name is bound in the body only
assert(same(sorted(SY.free(term(E.let_("a", null, E.variable("b"))), E.DIALECT)), ["b"])); // empty slots are skipped
assert(same(sorted(SY.free(term(E.quantifier("all", "x", E.variable("xs"), E.variable("x").gt(E.variable("y")))), E.DIALECT)), ["xs", "y"]));
assert(SY.imports(term(basic), E.DIALECT).length === 0);
for (const [error, name] of [[FE.NameError, "NameError"], [FE.ImportError, "ImportError"], [FE.IndexError, "IndexError"],
  [FE.OverflowError, "OverflowError"], [FE.ZeroDivisionError, "ZeroDivisionError"]] as const) {
  assert(new error("m") instanceof Error && new error("m").name === name); // Python's exceptions, by the same names
}
assert(same(sorted(SY.free(samples.Python3, P.DIALECT)), ["person"])); // np, f and a are bound by imports and the let
assert(same(sorted(SY.free(term(P.call("hasattr", P.name("x"), "k"), P.DIALECT), P.DIALECT)), ["x"])); // builtins are ambient
assert(same(sorted(SY.free(term(X.infix("+", X.cell("A1"), X.name("r")), X.DIALECT), X.DIALECT)), ["r"])); // cells are resolved by the workbook, not bound
assert(same(sorted(SY.free(term(M.import_("pkg.*", M.identifier("x")), M.DIALECT), M.DIALECT)), ["x"])); // MATLAB's imports bind functions, not variables
assert(same(SY.imports(samples.Python3, P.DIALECT).map((i) => F.kindOf(i)[0]), ["import", "importfrom"]));
assert(same(SY.imports(samples.Matlab, M.DIALECT).map((i) => F.nameOf(i)), ["pkg.*"]));

<a id="FRM-10"></a>
## FRM-10 · A dialect may extend another: its kinds are the base's and its own, added to its union by schema extension

In [ ]:
/** An extension of Patterns, added to its union: `twice(x)`, x + x; `import(module, body)`, an import binding the
 * module's name within its body; `cell`, a reference the scope resolves without a binding; and `apply(function,
 * arguments)`, a slot and variadic arguments. And `Bare`, a dialect of its own: its `Term`, `Expressions.Bare.Term`,
 * the framework's literal and the import, the branch `bare` of Term. */
function install(txn: any): any {
  const S = txn.Schemas;
  const Framework = txn.Expressions.Framework;
  return txn.uid("frm.Ext/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Term: S.OfUnion, Patterns: S.OfUnion, Bare: S.OfUnion, Twice: S.OfObject, Import: S.OfObject, Cell: S.OfObject,
      Apply: S.OfObject })
    .spec([
      (_: any) => _.Twice.name("frm.Twice").properties({ operand: _.Term, domain: S.OfStr }),
      (_: any) => _.Import.name("frm.Import").properties({ name: S.OfStr, body: _.Term, domain: S.OfStr }),
      (_: any) => _.Cell.name("frm.Cell").properties({ name: S.OfStr, domain: S.OfStr }),
      (_: any) => _.Apply.name("frm.Apply").properties({
        function: _.Term, arguments: S.OfContainer((c: any) => c.index([S.OfInt]).element([_.Term])), domain: S.OfStr }),
      (_: any) => _.Patterns.name("Expressions.Patterns.Term").branches([ // Patterns' union, extended
        (t: any) => t.name("twice").schema([_.Twice]), (t: any) => t.name("import").schema([_.Import]),
        (t: any) => t.name("cell").schema([_.Cell]), (t: any) => t.name("apply").schema([_.Apply])]),
      (_: any) => _.Bare.name("Expressions.Bare.Term").branches([
        (t: any) => t.name("literal").schema([Framework.Literal]), (t: any) => t.name("import").schema([_.Import])]),
      (_: any) => _.Term.name("Expressions.Framework.Term").branches([(t: any) => t.name("bare").schema([_.Bare])]),
    ])]))]);
}

store.transaction(install);
const Ext = new F.Dialect("Ext", {
  twice: new F.Kind(F.APPLICATION, { signature: new D.Function([BD.Int], BD.Int) }), // an open vocabulary: its tag names it
  import: new F.Kind(F.IMPORT, { required: ["name"], names: (attributes) => [attributes.get("name")] }),
  cell: new F.Kind(F.REFERENCE, { required: ["name"], lexical: false }),
  apply: new F.Kind(F.APPLICATION, { signature: new D.Opaque() }),
}, { domain_of: BD.of, extends: E.DIALECT });
assert(same([...Ext.kinds().keys()], [...E.DIALECT.kinds().keys(), "twice", "import", "cell", "apply"]) && Ext.base === E.DIALECT && E.DIALECT.base === null);
assert(Ext.tag === E.DIALECT.tag && Ext.union === E.DIALECT.union); // its terms are its base's union's
raises(ValueError, () => term(Ext.make(new F.Form("apply")), Ext), "an apply takes at least 1 arguments, got 0");
const twice = (operand: unknown) => new E.Writer((u: any) => u.twice((r: any) => r.operand(Ext.spec(operand))));
const import_ = (name: string, body: unknown, dialect: F.Dialect = Ext) => new F.Writer(
  (u: any) => u.import((r: any) => r.name(name).body(dialect.spec(body))), dialect.tag);
const cell = (name: string) => new F.Writer((u: any) => u.cell((r: any) => r.name(name)), Ext.tag);
x = E.variable("x");
const mixed = term(x.add(twice(x)), Ext); // a writer of the base holds the extension's term
assert(Ext.validate(mixed, ["x"]).length === 0 && F.kindOf(F.form(mixed).arguments[1])[0] === "twice");
assert(same(E.DIALECT.validate(mixed, ["x"]), ["argument 1: Patterns has no kind 'twice'"])); // the base reads only its own
raises(ValueError, () => V.OfAny(mixed, { x: 1n }), "Patterns has no kind 'twice'");

/** A scope with modules to import and cells to read. */
class Workbook extends SY.Variables {
  readonly cells: Map<string, unknown>;
  readonly modules: Map<string, unknown>;

  constructor(variables: SY.Bindings = {}, cells: Record<string, unknown> = {}, modules: Record<string, unknown> = {}) {
    super(variables);
    this.cells = new Map(Object.entries(cells));
    this.modules = new Map(Object.entries(modules));
  }

  override lookup(reference: any): any {
    if (F.kindOf(reference)[0] === "cell") return this.cells.get(F.nameOf(reference));
    return super.lookup(reference);
  }

  override enter(declaration: any): SY.Scope {
    const name = F.nameOf(declaration);
    return this.bind(name, this.modules.get(name));
  }
}

interpreter = new FV.Interpreter(Ext, new Map<string, any>([["call", V.OPERATIONS], ["quantifier", V.QUANTIFIERS],
  ["twice", (thunks: FV.Thunk[]) => thunks[0]!() * 2n]]), { scope: (variables) => new Workbook(variables) });
assert(interpreter.run(mixed, { x: 3n }) === 9n);
const program = term(import_("m", E.variable("m").get("k").add(cell("A1"))), Ext);
assert(interpreter.run(program, new Workbook({}, { A1: 1n }, { m: term(E.literal(1n)) })) === null); // no property k
assert(Ext.validate(program).length === 0 && SY.free(program, Ext).size === 0); // the import binds m; a cell needs no binding
assert(same(SY.imports(program, Ext).map((i) => F.nameOf(i)), ["m"]));
assert(Ext.infer(program).name() === "int | float" && Ext.infer(mixed, { x: BD.Int }) === BD.Int);
raises(NotImplementedError, () => new SY.Variables().enter(program), "Variables has no imports");
raises(ValueError, () => interpreter.run(term((u: any) => u.import((r: any) => r.name("m")), Ext)), "an import needs a body");
assert(same(Ext.validate(term((u: any) => u.import((r: any) => r.body(E.variable("m"))), Ext)), ["an import needs a name", "body: variable 'm' is not bound"]));
// Partial evaluation of an extension's terms: an import is rebuilt, binding its names within its body.
const reducer = new FP.Reducer(interpreter, (value) => (typeof value === "bigint" ? E.literal(value) : null), () => null);
const needsY = term(import_("m", E.variable("m").get("k").add(E.variable("y"))), Ext);
assert(F.same(term(reducer.run(needsY, { x: 1n }), Ext), needsY));
assert(F.same(term(reducer.run(term(twice(x).add(twice(E.variable("y"))), Ext), { x: 2n }), Ext), term(E.literal(4n).add(twice(E.variable("y"))), Ext)));
// A dialect's kinds need no checks, domains or names of their own.
const Bare = new F.Dialect("Bare", { literal: new F.Kind(F.LITERAL), import: new F.Kind(F.IMPORT) }, { domain_of: BD.of });
const three = (u: any) => u.literal((r: any) => r.value((p: any) => p.as_scalar(3n))); // Bare has no writers
const bare = term(import_("m", three, Bare), Bare);
assert(Bare.validate(bare).length === 0 && SY.free(bare, Bare).size === 0 && Bare.infer(bare) === BD.Int);
assert(new FV.Interpreter(Bare, new Map()).run(term(three, Bare)) === 3n);
raises(ValueError, () => new F.Dialect("Clash", { literal: new F.Kind(F.APPLICATION) }, { domain_of: BD.of, extends: E.DIALECT }),
  "Clash would declare 'literal', which Patterns has");
raises(ValueError, () => new F.Dialect("Clash", { error: new F.Kind(F.APPLICATION) }, { domain_of: BD.of }),
  "Clash would declare 'error', which the framework has"); // every dialect reads invalid syntax

<a id="FRM-11"></a>
## FRM-11 · Invalid syntax, `error` and `missing`: every dialect's `Term` holds them, validation reports them, evaluation refuses them by name, and printers write them as written

In [ ]:
const PRINTERS = new Map<F.Dialect, (expression: unknown) => string>([[P.DIALECT, Python3.Text.ToText],
  [M.DIALECT, Matlab.Text.ToText], [X.DIALECT, Excel.Text.ToText], [L.DIALECT, Latex.Text.ToText],
  [C.DIALECT, CDialect.Text.ToText], [CPP.DIALECT, CppDialect.Text.ToText], [SV.DIALECT, SystemVerilog.Text.ToText]]); // Patterns has none
for (const [dialect, write] of PRINTERS) assert(write(holders.get(dialect)) === write(samples[dialect.name()])); // FRM-04's
const error = (dialect: F.Dialect) => (u: any) => u.error((r: any) => r.text("(a +* 1)").terms([ // what a parser skipped
  dialect.spec((v: any) => v.variable((w: any) => w.name("a")))])); // and the terms it recovered, of its dialect
const missing = (u: any) => u.missing((r: any) => r.kind("expression")); // what it expected
for (const dialect of DIALECTS) {
  const [skipped, expected] = [term(error(dialect), dialect), term(missing, dialect)];
  assert(F.form(skipped).kind === "error" && equal(F.form(skipped).attributes, map({ text: "(a +* 1)" })));
  assert(F.form(skipped).arguments.length === 1 && form_equal(F.form(expected), new F.Form("missing", { kind: "expression" })));
  assert(same(dialect.validate(skipped), ["invalid syntax '(a +* 1)'"]) && same(dialect.validate(expected), ["missing expression"]));
  assert(same(dialect.validate(term((u: any) => u.error((r: any) => r), dialect)), ["invalid syntax"]));
  assert(same(dialect.validate(term((u: any) => u.missing((r: any) => r), dialect)), ["missing a term"]));
  assert(F.same(term(dialect.make(F.form(skipped)), dialect), skipped)); // forms and make, as for every kind
  assert(same([skipped, expected].map((t) => dialect.kind(t)[2].role), [F.INVALID, F.INVALID]));
  const evaluate = EVALUATORS.get(dialect) as ((expression: unknown) => unknown) | undefined;
  if (dialect === SV.DIALECT) { // SystemVerilog's evaluator validates first, as it sizes its terms
    raises(ValueError, () => SystemVerilog.Evaluators.OfAny(skipped), "cannot evaluate an invalid expression: invalid syntax '(a +* 1)'");
  } else if (evaluate !== undefined) {
    raises(ValueError, () => evaluate(skipped), `${dialect.name()} error is not evaluated`);
    raises(ValueError, () => evaluate(expected), `${dialect.name()} missing is not evaluated`);
  }
  const print = PRINTERS.get(dialect);
  if (print !== undefined) { // an error as the source it spans, a missing term as nothing
    assert(print(skipped) === (dialect === X.DIALECT ? "=(a +* 1)" : "(a +* 1)"));
    assert(print(expected) === (dialect === X.DIALECT ? "=" : ""));
    raises(ValueError, () => print(term((u: any) => u.error((r: any) => r), dialect)), `${dialect.name()} error has no text to print`);
  }
}
assert(Python3.Text.ToText(term(P.binop("+", P.name("a"), missing), P.DIALECT)) === "a + "); // within a term, as written
assert(same(sorted(SY.free(term(error(E.DIALECT)), E.DIALECT)), ["a"])); // the terms recovered within an error are terms
raises(ValueError, () => Translators.between(P.DIALECT, M.DIALECT).forward(term(error(P.DIALECT), P.DIALECT)),
  "Python3 error has no Matlab counterpart"); // nothing translates invalid syntax
raises(ValueError, () => E.DIALECT.infer(term(missing)), "cannot infer the domain of an invalid expression: missing expression");

<a id="FRM-12"></a>
## FRM-12 · A dialect reads only its own terms: a term holds its dialect, a shared kind's slot holds any dialect's term, and validation reports another's; C and C++ each define the kinds they share, each recording its standards

In [ ]:
const foreign = term(P.let_("a", E.variable("a"), P.name("a")), P.DIALECT); // a shared kind's slot: Patterns' variable in it
assert(F.dialectOf(F.form(foreign).arguments[0]) === E.DIALECT.union && F.dialectOf(foreign) === P.DIALECT.union);
assert(same(P.DIALECT.validate(foreign), ["value: a Patterns variable is not a Python3 term"]));
raises(ValueError, () => Python3.Evaluators.OfAny(foreign), "a Patterns variable is not a Python3 term");
raises(ValueError, () => Matlab.Text.ToText(term(X.function("AND", true), X.DIALECT)), "an Excel call is not a Matlab term");
assert(!F.same(term(E.variable("a")), term(P.name("a"), P.DIALECT))); // the same kind, in another dialect
const own = term(P.binop("+", P.name("a"), 1n), P.DIALECT); // a dialect's own kind's slot: its union's terms alone
assert(same(F.form(own).arguments.map((a) => F.dialectOf(a)), [P.DIALECT.union, P.DIALECT.union]));
assert((F.form(own).arguments[0] as any).as_schema().name === P.DIALECT.union); // one level: its kind alone
raises(Error, () => term(P.binop("+", X.cell("A1"), 1n), P.DIALECT), "'cell' is not a member of a union value"); // the schema refuses a kind its union lacks
// C and C++: each its own kinds, of the same shapes, and its own Term; each Kind records its language's standards
assert(same(C.STANDARDS, ["C11", "C17", "C23"]) && same(CPP.STANDARDS, ["C++17", "C++20", "C++23"]));
raises(TypeError, () => CPP.DIALECT.spec(new Uint8Array([120])), "expected a term, a writer or a native value, got b'x'"); // no constant of C++ holds bytes
assert([...C.DIALECT.kinds()].every(([tag, kind]) => INVALID.has(tag) || same(kind.standards, C.STANDARDS)));
assert([...CPP.DIALECT.kinds()].every(([tag, kind]) => INVALID.has(tag) || same(kind.standards, CPP.STANDARDS)));
assert(DIALECTS.every((dialect) => [...dialect.kinds()].every(([tag, kind]) =>
  !(INVALID.has(tag) || (dialect !== C.DIALECT && dialect !== CPP.DIALECT && dialect !== P.DIALECT)) || kind.standards.length === 0)));
// Python3: each kind records the versions of Python 3 that have it, from the one it arrived in; an extension, none
assert(P.STANDARDS[0] === "Python 3.0" && P.STANDARDS[P.STANDARDS.length - 1] === "Python 3.15" && P.STANDARDS.length === 16);
const python3Kinds = P.DIALECT.kinds();
assert(same(python3Kinds.get(P.BINOP)!.standards, P.STANDARDS) && same(python3Kinds.get(P.NAMEDEXPR)!.standards, P.STANDARDS.slice(8)));
assert(same(python3Kinds.get(P.TEMPLATESTR)!.standards, ["Python 3.14", "Python 3.15"]) && python3Kinds.get(P.AWAIT)!.standards[0] === "Python 3.5");
assert(same([...P.EXTENSIONS], ["let", "import", "importfrom"])
  && [...P.EXTENSIONS, ...INVALID].every((tag) => python3Kinds.get(tag)?.standards.length === 0));
assert(CDialect.Text.ToText(samples.C) === CppDialect.Text.ToText(samples.Cpp)
  && CDialect.Text.ToText(samples.C) === "!person.a && person->b[0] > 1.5f ? (uint8_t)f(1) : 5u");
assert(CDialect.Evaluators.OfAny(term(C.binary("+", 1n, 2n), C.DIALECT)) === 3n
  && CppDialect.Evaluators.OfAny(term(CPP.binary("+", 1n, 2n), CPP.DIALECT)) === 3n);
assert(same(C.DIALECT.validate(term(C.call("f", CPP.identifier("x")), C.DIALECT), ["x"]), [
  "argument 0: a Cpp variable is not a C term"])); // a shared kind's slot holds any dialect's term; validation tells them apart